# Growth-rate comparison: GS2 and TGLF

Compare growth rate versus safety factor at a selected binormal wavenumber. This short example follows the `scan_q/S1_3` comparison in `Linear_Plotting.ipynb`, using saved PyroScan outputs. No simulations are launched or source data written.

GS2 points with growth-rate tolerance ≥ 0.3 are masked, following the reference notebook. TGLF uses mode 0. Nearest wavenumbers are reported separately for both models; this is a comparison of the selected samples, not interpolation onto a common grid.

## Imports and settings

Set GK_DATA_ROOT in the repository's local.env. Edit relative scan paths and scientific choices here. Use a kernel with compatible Pyrokinetics and python-dotenv installations.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv
import matplotlib.pyplot as plt
from pyrokinetics import PyroScan

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file() and (path / "notebooks").is_dir()
)
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)

analysis_name = "growth_rate_comparison"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
codes = ["GS2", "TGLF"]
run_template = "Runs"
project = "scan_q"
case = "S1_3"
scan_information = {
    "GS2": "parameter_scan_GS2",
    "TGLF": "parameter_scan_TGLF_sat2",
}  # Omit a code or use an empty string when there is no scan subdirectory.
metadata_file = "pyroscan.json"
output_file = "pyroscan.nc"

ky = 0.3
reference_q = 3.49
quality_max = 0.3

## Load existing scans

Pyrokinetics handles normalisation. This cell requires the scan files and the base inputs referenced by `pyroscan.json`. Rerun it only when inputs change. Leave warnings visible: the supplied GS2 example currently reports inconsistent beta/reference values, so loading successfully does not establish physical consistency.

In [ ]:
scans = {}
for code in codes:
    directory = data_root / code / run_template / project / case / scan_information.get(code, "")
    scan = PyroScan(pyroscan_json=directory / metadata_file, load_base_pyro=True)
    scan.load_gk_output(netcdf_file=directory / output_file)
    scans[code] = scan

## Select and filter

The strict tolerance mask is identical to the reference notebook. No extra reduction or filtering is applied to TGLF. Labels use the Pyrokinetics normalisation (growth rate in units of $c_s/a$).

In [ ]:
gs2 = scans["GS2"].gk_output["growth_rate"].sel(ky=ky, method="nearest")
quality = scans["GS2"].gk_output["growth_rate_tolerance"].sel(ky=ky, method="nearest")
gs2 = gs2.where(quality < quality_max)
tglf = scans["TGLF"].gk_output["growth_rate"].sel(ky=ky, method="nearest").sel(mode=0)
print(f"Requested ky={ky:g}; GS2 ky={float(gs2.ky):g}; TGLF ky={float(tglf.ky):g}")

## Plot

Shared defaults come from `paper.mplstyle`. GS2 is explicitly black as the gyrokinetic reference against TGLF. TGLF uses the colour cycle here; no universal TGLF/GFTM colours are prescribed. Edit labels, limits, or individual curve choices here; rerunning this cell creates a fresh figure.

In [ ]:
fig, ax = plt.subplots()
gs2.plot(ax=ax, label=f"GS2, ky={float(gs2.ky):g}", color="black")
tglf.plot(ax=ax, label=f"TGLF, ky={float(tglf.ky):g}", linestyle="--")
ax.axvline(reference_q, color="tab:red", linestyle="--", marker="", alpha=0.3)
ax.set(xlabel=r"$q$", ylabel=r"$\gamma\,a/c_s$", title=f"{case}: growth rate vs q")
ax.legend()
plt.show()

## Save

Export resolution and padding come from the shared style. Rerunning replaces this PNG.

In [ ]:
output_dir = ROOT / "Plots" / analysis_name
output_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(output_dir / f"{case}_ky_{ky:g}.png")

## Reading the comparison

The black curve shows GS2 after the tolerance mask; the dashed curve shows TGLF mode 0. Gaps may reflect filtered or missing values. The red line marks the reference safety factor from the original notebook. This example demonstrates the workflow; it is not a validation of model agreement, convergence, or the physical reference values.